In [1]:
!python --version

Python 3.10.12


In [ ]:
# Verify we are connected to the GPU Runtime so we are able to use GPU
!nvidia-smi

In [ ]:
# Install the Required Packages
!pip install -q -U bitsandbytes --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/transformers.git --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/peft.git --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/accelerate.git --use-deprecated=legacy-resolver
!pip install -q -U datasets scipy ipywidgets matplotlib --use-deprecated=legacy-resolver
!pip install accelerate

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Now we are extract the data from my Amazon S3 Bucket, then we start training our chatbot for stackoverflow question and answers
!pip install boto3
import os
import boto3
from botocore.exceptions import NoCredentialsError

aws_access_key_id = os.environ['AWS_ACCESS_KEY_ID']  # AWS S3 bucket Access Key ID
aws_secret_access_key = os.environ['AWS_SECRET_ACCESS_KEY'] # AWS S3 bucket Secret Access Key
aws_region = 'us-east-1'

# Create a session using my AWS credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

In [ ]:
# Define my S3 bucket name and folder
bucket_name = 'archistackoverflow'
s3_folder = ''

# Create an S3 client
s3 = session.client('s3')

# List files in the specified my S3 folder
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_folder)

if 'Contents' in response:
    files = [content['Key'] for content in response['Contents']]
    print("Files in S3 bucket:")
    for file in files:
        print(file)
else:
    print("No files found in the S3 bucket or folder.")

In [ ]:
import os

# Create a local directory inside my colab drive to store downloaded files
local_dir = '/content/s3_downloads'
os.makedirs(local_dir, exist_ok=True)

# Download each file from S3
for file_key in files:
    local_file_path = os.path.join(local_dir, os.path.basename(file_key))
    try:
        s3.download_file(bucket_name, file_key, local_file_path)
        print(f"Downloaded {file_key} to {local_file_path}")
    except NoCredentialsError:
        print("Credentials not available")
    except FileNotFoundError:
        print(f"The file {file_key} was not found in S3")


In [ ]:
# Change directory to my dataset directory
%cd /content/s3_downloads

In [ ]:
from datasets import load_dataset

bot_train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
bot_eval_dataset = load_dataset('json', data_files='eval.jsonl', split='train')

In [ ]:
def formatting_func(example):
    text = f"### Question: {example['prompt']}\n ### Answer: {example['response']}"   ## Here prompt and response are keys in jsonl files
    return text

In [ ]:
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])  # My Access Token Hugging Face
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_model_id = "mistralai/Mistral-7B-v0.1"   # My Base Model For Train
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    padding_side="left",
    add_eos_token=True,
    add_bos_token=True,
)
tokenizer.pad_token = tokenizer.eos_token

def generate_and_tokenize_prompt(prompt):
    return tokenizer(formatting_func(prompt))

In [ ]:
tokenized_train_dataset = bot_train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = bot_eval_dataset.map(generate_and_tokenize_prompt)

In [ ]:
import matplotlib.pyplot as plt

def plot_data_lengths(tokenized_train_dataset, tokenized_val_dataset):
    lengths = [len(x['input_ids']) for x in tokenized_train_dataset]
    lengths += [len(x['input_ids']) for x in tokenized_val_dataset]
    print(len(lengths))

    # Plotting the histogram
    plt.figure(figsize=(10, 6))
    plt.hist(lengths, bins=20, alpha=0.7, color='blue')
    plt.xlabel('Length of input_ids')
    plt.ylabel('Frequency')
    plt.title('Distribution of Lengths of input_ids')
    plt.show()

plot_data_lengths(tokenized_train_dataset, tokenized_val_dataset)

In [ ]:
max_length = 512 # This was an appropriate max length for my dataset

def generate_and_tokenize_prompt2(prompt):
    result = tokenizer(
        formatting_func(prompt),
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
tokenized_train_dataset = bot_train_dataset.map(generate_and_tokenize_prompt2)
tokenized_val_dataset = bot_eval_dataset.map(generate_and_tokenize_prompt2)

In [ ]:
print(tokenized_train_dataset[1]['input_ids'])

In [ ]:
plot_data_lengths(tokenized_train_dataset, tokenized_val_dataset)

In [ ]:
eval_prompt = "I need to find a loan for a house that meets my specific needs. What criteria should I consider?"

In [ ]:
# Init an eval tokenizer that doesn't add padding or eos token
eval_tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    add_bos_token=True,
)

model_input = eval_tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(eval_tokenizer.decode(model.generate(**model_input, max_new_tokens=256, repetition_penalty=1.15)[0], skip_special_tokens=True))

In [ ]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
print(model)

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

In [ ]:
print(model)

In [ ]:
if torch.cuda.device_count() > 1: # If more than 1 GPU for parellel training
    model.is_parallelizable = True
    model.model_parallel = True

In [ ]:
from accelerate import Accelerator

# Initialize the accelerator
accelerator = Accelerator()

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 169, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/req_command.py", line 242, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 92, in resolve
    result = self._result = resolver.resolve(
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
  File "/usr/local/li

In [ ]:
model = accelerator.prepare_model(model)

In [ ]:
import transformers
from datetime import datetime

project = "stackoverflow-finetune"
base_model_name = "mistral"
run_name = base_model_name + "-" + project
output_dir = "./" + run_name

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1,
        gradient_checkpointing=True,
        max_steps=500,
        learning_rate=2.5e-5, # Want a small lr for finetuning
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=25,              # When to start reporting loss
        logging_dir="./logs",        # Directory for storing logs
        save_strategy="steps",       # Save the model checkpoint every logging step
        save_steps=25,                # Save checkpoints every 50 steps
        evaluation_strategy="steps", # Evaluate the model every logging step
        eval_steps=25,               # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
      #  report_to="wandb",           # Comment this out if you don't want to use weights & baises
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}"          # Name of the W&B run (optional)
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()